# Diagnóstico posterior da matheurística determinística

Este notebook **somente lê resultados**. Não instancia nem resolve modelos de otimização e não é executado pelo pipeline.
Escolha uma execução por `run_id`; índices de produto, veículo e período começam em **zero**, e o depósito é o nó **0**.
`u` e `eta` são gravadas sempre que houver uma solução na etapa. Variáveis físicas intermediárias e auxiliares de metas exigem `solver.matheuristic.debug.enabled=true`.
Zeros omitidos de uma família marcada `sparse=true` são zeros. Uma família ausente **não** é uma matriz de zeros: significa que ela não foi salva.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(REPO))
from src.solvers._solver_common import ProblemData, solution_components, evaluate_configured_objective
from src.solvers._routing_common import routes_from_z, ordered_arcs, validate_variables
from src.helpers.ReadPrpFile import ReadPrpFile

OUTPUT_ROOT = REPO / "out"  # Ajuste para o diretório dos resultados.
RUN_ID = 'f0ee4342f6-matheuristic-beec215390'               # Copie um run_id da tabela abaixo e execute novamente.
INSTANCE_FILE = None        # Opcional: novo caminho para o .dat original.
metadata_files = sorted(OUTPUT_ROOT.rglob("*.metadata.json"))
metadata_files = [p for p in metadata_files if p.parent.name == "matheuristic"]
inventory = pd.DataFrame([{"run_id":p.name.removesuffix(".metadata.json"),"path":str(p)} for p in metadata_files])
display(inventory)

,run_id,path
0,f0ee4342f6-matheuristic-beec215390,/Users/gabamaro/.codex/worktrees/d672/lsvrp/ou...


## Carregamento e comparação das etapas

A tabela distingue `objective` (objetivo próprio da etapa) e `configured_objective` (critério para selecionar a solução).
O objetivo da melhoria é transporte; seu gap e bound pertencem à etapa `route_improvement`.
O gap de `restricted` descreve apenas a região limitada pela ordem. Nenhum deles certifica o ótimo do MPPRP irrestrito.
Em um TSP recuperado do cache, `cache_hit=true` e os tempos da execução atual são zero; bound/gap descrevem a resolução que preencheu o cache.

In [2]:
assert RUN_ID, "Escolha RUN_ID na célula anterior."
paths = inventory.loc[inventory.run_id.eq(RUN_ID),"path"].tolist()
assert len(paths) == 1, "run_id não encontrado ou duplicado."
meta_path = Path(paths[0])
meta = json.loads(meta_path.read_text())
variables_df = pd.read_parquet(meta_path.parent/f"{RUN_ID}.variables.parquet")
summary = pd.read_parquet(meta_path.parent/f"{RUN_ID}.stages.parquet")
catalog = meta["available_variables"]

silent = SimpleNamespace(debug=lambda *args:None)
data = meta.get("parameters")
if data is None:
    # O .dat original permite auditar a solução final mesmo sem debug intermediário.
    instance_path = Path(INSTANCE_FILE or meta["instance_file"])
    if not instance_path.is_absolute():
        instance_path = REPO / instance_path
    assert instance_path.exists(), "Ajuste INSTANCE_FILE para localizar o .dat original."
    data = ReadPrpFile(str(instance_path),silent).getDataSet()
    data.update(weight=meta["weight"],alpha=meta["alpha"],targets=meta["targets"],
                strengthened_bounds=meta["strengthened_bounds"])
problem = ProblemData.from_map(data)
config = meta["config"]
pi = meta["order"]
display(summary)
print("Etapa selecionada:",meta["selected_stage"],"Ordem:",pi)
fig = px.bar(summary.melt(id_vars="stage",value_vars=[c for c in
              ["build_seconds","solve_seconds","elapsed_seconds"] if c in summary],
              var_name="medida",value_name="segundos"),x="stage",y="segundos",color="medida",barmode="group")
fig.show()  # elapsed inclui solve/build; as barras não são parcelas somáveis.

,run_id,hash_file,stage,status,bound,gap,cache_hit,solve_seconds,build_seconds,variables,...,gap_scope,selected,extract_seconds,has_incumbent,has_solution,validation_errors,features,rounded_capacity,configured_objective,costs
0,f0ee4342f6-matheuristic-beec215390,d3632e1fc7c0a01cd42177c98aedf0da35458c2d,tsp,integer optimal solution,2024.000000,0.000000,False,0.013691,0.026670,30,...,tsp,False,NaN,None,None,None,None,None,NaN,None
1,f0ee4342f6-matheuristic-beec215390,d3632e1fc7c0a01cd42177c98aedf0da35458c2d,restricted,"integer optimal, tolerance",0.422189,0.000092,None,0.102213,0.007025,520,...,restricted,True,0.000100,True,True,[],"{'coelho_inequalities': False, 'hc1': False, '...","{'callback_calls': 0.0, 'callback_errors': 0.0...",0.422228,"[[27139.999999999996, 36250.0, 5190.0], [1100...."
2,f0ee4342f6-matheuristic-beec215390,d3632e1fc7c0a01cd42177c98aedf0da35458c2d,route_improvement,integer optimal solution,33280.000000,0.000000,None,0.013444,0.007096,496,...,route_improvement,False,0.000086,True,True,[],"{'coelho_inequalities': False, 'hc1': False, '...","{'callback_calls': 0.0, 'callback_errors': 0.0...",0.422228,"[[27139.999999999996, 36250.0, 5190.0], [1100...."


Etapa selecionada: restricted Ordem: [0, 1, 2, 3, 4, 5, 0]


In [3]:
def restore(stage,var):
    spec = catalog.get(stage,{}).get(var)
    if spec is None:
        raise KeyError(f"{stage}/{var} não foi salva; habilite debug para estados intermediários.")
    array = np.zeros(spec["shape"],dtype=np.float64)
    rows = variables_df.loc[variables_df.stage.eq(stage)&variables_df["var"].eq(var)]
    if not spec["indices"]:
        return np.array(rows.value.iloc[0] if len(rows) else 0.)
    for row in rows.itertuples(index=False):
        key = tuple(int(getattr(row,index)) for index in spec["indices"])
        array[key] = row.value
    return array

physical = ("X","Y","I","Q","R","Z")
states = {stage:{var:restore(stage,var) for var in (*physical,"eta") if var in families}
          for stage,families in catalog.items() if all(var in families for var in physical)}
# Recupera também a saída final no contrato legado, identificada pelo mesmo hash.
final_path = meta_path.parent.parent/"parquets"/f"{meta['hash_file'][:6]}_fobs.parquet"
stamp = meta.get('final_output')
final_available = (final_path.exists() and stamp is not None and
                   final_path.stat().st_mtime_ns == stamp['mtime_ns'] and final_path.stat().st_size == stamp['size'])
if final_available:
    final_df = pd.read_parquet(final_path)
    assert final_df.empty or set(final_df.hash_file) == {meta["hash_file"]}, "Arquivo final não corresponde à execução."
    shapes = {"X":(problem.p,problem.t),"Y":(problem.p,problem.t),
              "I":(problem.p,problem.i,problem.t),"Q":(problem.p,problem.v,problem.i,problem.t),
              "R":(problem.p,problem.v,problem.i,problem.k,problem.t),"Z":(problem.v,problem.i,problem.k,problem.t)}
    indices = {"X":("p","t"),"Y":("p","t"),"I":("p","i","t"),"Q":("p","v","i","t"),
               "R":("p","v","i","k","t"),"Z":("v","i","k","t")}
    if all(var in set(final_df.get('var',pd.Series(dtype=str))) for var in physical):
        state = {var:np.zeros(shape) for var,shape in shapes.items()}
        for var,keys in indices.items():
            for row in final_df.loc[final_df['var'].eq(var)].itertuples(index=False):
                state[var][tuple(int(getattr(row,key)) for key in keys)] = row.value
        states['final'] = state
if final_path.exists() and not final_available:
    print("Saída final ausente desta execução ou sobrescrita; use os snapshots de debug.")
print("Estados físicos disponíveis:",list(states))

Estados físicos disponíveis: ['restricted', 'route_improvement', 'final']


## TSP, visitas, rotas e cargas

A ordem do TSP é comum aos produtos, veículos e períodos. A tabela de rotas mostra subsequências na etapa restrita e sequências livres na melhoria.
Escolha `STAGE` e `PERIOD` para inspeção. Os gráficos usam as coordenadas da instância; sem coordenadas, empregam uma disposição circular apenas para visualização, sem alterar custos.

In [7]:
def plot_routes(routes, title):
    coord = data.get('coordXY')
    if coord:
        x, y = np.asarray(coord['x']), np.asarray(coord['y'])
    else:
        angle = np.linspace(0, 2 * np.pi, problem.i, endpoint=False)
        x, y = np.cos(angle), np.sin(angle)

    fig = go.Figure()
    palette = px.colors.qualitative.Plotly
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        nodes = [0, *route, 0]
        color = palette[vehicle % len(palette)]
        fig.add_trace(
            go.Scatter(
                x=x[nodes],
                y=y[nodes],
                mode='lines+markers',
                line={'color': color, 'width': 2},
                marker={'size': 8},
                name=f'veículo {vehicle}',
                text=[f'parada {step}: nó {node}' for step, node in enumerate(nodes)],
                hovertemplate='%{text}<extra>%{fullData.name}</extra>',
            )
        )
        for step, (origin, destination) in enumerate(zip(nodes, nodes[1:]), start=1):
            dx, dy = x[destination] - x[origin], y[destination] - y[origin]
            # A seta aponta no sentido percorrido; o número identifica a perna da rota.
            fig.add_annotation(
                x=x[origin] + 0.68 * dx,
                y=y[origin] + 0.68 * dy,
                ax=x[origin] + 0.43 * dx,
                ay=y[origin] + 0.43 * dy,
                xref='x', yref='y', axref='x', ayref='y',
                text='', showarrow=True, arrowhead=3, arrowsize=1.2,
                arrowwidth=2, arrowcolor=color,
            )
            fig.add_annotation(
                x=x[origin] + 0.38 * dx,
                y=y[origin] + 0.38 * dy,
                text=str(step), showarrow=False,
                font={'color': color, 'size': 11},
                bgcolor='white', bordercolor=color, borderwidth=1,
            )

    fig.add_trace(
        go.Scatter(
            x=x, y=y, mode='markers+text', text=list(range(problem.i)),
            textposition='top center', name='nós',
            marker={'color': 'black', 'size': 10},
        )
    )
    fig.update_layout(
        title=title,
        yaxis_scaleanchor='x',
        annotations=fig.layout.annotations,
        legend_title_text='Ordem por veículo',
    )
    return fig

u = restore('tsp','u')
assert np.allclose(u,np.rint(u))
assert all(u[a,b] == 1 for a,b in zip(pi,pi[1:]) if a != b)
plot_routes([pi[1:-1]],'TSP: ordem comum').show()
route_records = []
for stage,state in states.items():
    try:
        for t,period in enumerate(routes_from_z(state['Z'])):
            for v,route in enumerate(period):
                route_records.append({'stage':stage,'t':t,'v':v,'route':[0,*route,0] if route else [],
                                      'sequence':' → '.join(map(str,[0,*route,0])) if route else '',
                                      'delivered':float(state['Q'][:,v,:,t].sum())})
    except ValueError as error:
        print(stage,'Falha de conectividade:',error)
display(pd.DataFrame(route_records))

STAGE = next(iter(states),None)  # Ajuste para restricted, route_improvement ou final.
PERIOD = 1
if STAGE:
    state = states[STAGE]
    routes = routes_from_z(state['Z'])[PERIOD]
    plot_routes(routes,f'{STAGE}, período {PERIOD}').show()
    rows = []
    for v,route in enumerate(routes):
        for i,k in zip([0,*route],[*route,0]) if route else []:
            rows.append({'v':v,'i':i,'k':k,'Z':state['Z'][v,i,k,PERIOD],
                         'carga_por_produto':state['R'][:,v,i,k,PERIOD].tolist(),
                         'carga_total':state['R'][:,v,i,k,PERIOD].sum()})
    display(pd.DataFrame(rows))

,stage,t,v,route,sequence,delivered
0,restricted,0,0,"[0, 4, 0]",0 → 4 → 0,276.0
1,restricted,0,1,"[0, 2, 3, 5, 0]",0 → 2 → 3 → 5 → 0,277.0
2,restricted,1,0,"[0, 1, 2, 0]",0 → 1 → 2 → 0,300.0
3,restricted,1,1,"[0, 3, 4, 5, 0]",0 → 3 → 4 → 5 → 0,300.0
4,restricted,2,0,"[0, 3, 5, 0]",0 → 3 → 5 → 0,173.0
5,restricted,2,1,[],,0.0
6,route_improvement,0,0,"[0, 4, 0]",0 → 4 → 0,276.0
7,route_improvement,0,1,"[0, 2, 3, 5, 0]",0 → 2 → 3 → 5 → 0,277.0
8,route_improvement,1,0,"[0, 1, 2, 0]",0 → 1 → 2 → 0,300.0
9,route_improvement,1,1,"[0, 3, 4, 5, 0]",0 → 3 → 4 → 5 → 0,300.0


,v,i,k,Z,carga_por_produto,carga_total
0,0,0,1,1.0,"[154.00000000000003, 145.99999999999997]",300.0
1,0,1,2,1.0,"[61.00000000000003, 100.99999999999997]",162.0
2,0,2,0,1.0,"[0.0, 0.0]",0.0
3,1,0,3,1.0,"[81.00000000000003, 218.99999999999997]",300.0
4,1,3,4,1.0,"[20.00000000000003, 108.0]",128.0
5,1,4,5,1.0,"[20.00000000000003, 98.0]",118.0
6,1,5,0,1.0,"[0.0, 0.0]",0.0


## Auditoria numérica

A auditoria verifica domínios, balanços de estoque/fluxo, ligação carga/arco, capacidade de produção/estoque e entrega sem visita.
A conectividade é verificada inclusive para visitas com entrega zero. Não é usada tolerância para classificar uma entrega positiva da etapa 3; a auditoria de resíduos usa tolerância numérica.
A tabela explicita os resíduos máximos, inclusive `eta/Z`, e os custos são recalculados com a mesma avaliação usada no pipeline.

In [5]:
audits,cost_records = [],[]
for stage,state in states.items():
    report = validate_variables(problem,state,ordered_arcs(pi) if stage == 'restricted' else None)
    previous = np.concatenate((np.asarray(problem.I_p_i_0)[:,:,None],state['I'][:,:,:-1]),axis=2)
    plant_res = previous[:,0]+state['X']-state['Q'][:,:,1:].sum(axis=(1,2))-state['I'][:,0]
    customer_res = previous[:,1:]+state['Q'][:,:,1:].sum(axis=1)-problem.d_p_i_t-state['I'][:,1:]
    flow_res = state['R'].sum(axis=2)[:,:,1:]-state['R'].sum(axis=3)[:,:,1:]-state['Q'][:,:,1:]
    eta_out = eta_in = np.nan
    if 'eta' in state:
        eta_out = np.max(np.abs(state['Z'].sum(axis=2)-state['eta']))
        eta_in = np.max(np.abs(state['Z'].sum(axis=1)-state['eta']))
    audits.append({'stage':stage,**report,'plant_residual':np.max(np.abs(plant_res)),
                   'customer_residual':np.max(np.abs(customer_res)), 'flow_residual':np.max(np.abs(flow_res)),
                   'eta_out_residual':eta_out,'eta_in_residual':eta_in,
                   'capacity_excess':np.maximum(state['R'].sum(axis=0)-problem.C*state['Z'],0).max()})
    costs = solution_components(problem,state)
    cost_records.append({'stage':stage,'configured_objective':evaluate_configured_objective(problem,state,config),
                         **{f'F{j+1}':float(costs[j].sum()) for j in range(5)}})
display(pd.DataFrame(audits))
display(pd.DataFrame(cost_records))
if 'restricted' in states and 'route_improvement' in states:
    before,after = states['restricted'],states['route_improvement']
    display(pd.DataFrame([{'quantity':name,'max_change':np.max(np.abs(before[name]-after[name]))} for name in ('X','Y','I')]
        +[{'quantity':'Q agregado','max_change':np.max(np.abs(before['Q'].sum(axis=1)-after['Q'].sum(axis=1)))}]))
    for t in range(problem.t):
        active = after['Q'].sum(axis=(0,1))[:,t] > 0
        visited = after['Z'].sum(axis=(0,2))[:,t] > .5
        assert np.array_equal(active[1:],visited[1:]), f'Visitas indevidas na melhoria, período {t}'

,stage,feasible,violations,plant_residual,customer_residual,flow_residual,eta_out_residual,eta_in_residual,capacity_excess
0,restricted,True,[],0.0,2.842171e-14,0.0,0.0,0.0,0.0
1,route_improvement,True,[],0.0,2.842171e-14,0.0,0.0,0.0,0.0
2,final,True,[],0.0,2.842171e-14,0.0,NaN,NaN,0.0


,stage,configured_objective,F1,F2,F3,F4,F5
0,restricted,0.422228,68580.0,2500.0,3499.0,27720.0,5560.0
1,route_improvement,0.422228,68580.0,2500.0,3499.0,27720.0,5560.0
2,final,0.422228,68580.0,2500.0,3499.0,27720.0,5560.0


,quantity,max_change
0,X,0.000000e+00
1,Y,7.771561e-16
2,I,0.000000e+00
3,Q agregado,0.000000e+00


## Produção, estoques e entregas

As quantidades são contínuas. Escolha um produto e período para inspecionar seus valores fracionários.
As tabelas de `P`, `N` e `lambda` mostram os auxiliares efetivamente utilizados pelo solver quando o debug os salvou.

In [6]:
PRODUCT = 0
if STAGE:
    state = states[STAGE]
    production = pd.DataFrame({'t':range(problem.t),'X':state['X'][PRODUCT],'Y':state['Y'][PRODUCT]})
    display(production)
    px.line(production,x='t',y='X',markers=True,title=f'Produção: {STAGE}, produto {PRODUCT}').show()
    px.imshow(state['I'][PRODUCT],labels={'x':'período','y':'nó','color':'estoque'},title=f'Estoque: produto {PRODUCT}').show()
    delivery = state['Q'][PRODUCT,:,:,PERIOD]
    display(pd.DataFrame(delivery,index=[f'veículo {v}' for v in range(problem.v)]))
    px.imshow(delivery,labels={'x':'nó','y':'veículo','color':'entrega'},title=f'Entregas: período {PERIOD}').show()
for stage,families in catalog.items():
    for var in ('P','N','lambda'):
        if var in families:
            print(stage,var)
            display(restore(stage,var))

,t,X,Y
0,0,342.0,1.0
1,1,235.0,1.0
2,2,173.0,1.0


,0,1,2,3,4,5
veículo 0,0.0,0.0,0.0,0.0,115.0,0.0
veículo 1,0.0,0.0,56.0,83.0,0.0,88.0


restricted P


array([[ 5312.        , 14992.        ,     0.        ],
       [  737.771261  ,   713.5483871 ,     0.        ],
       [  243.7       ,  1377.4       ,     0.        ],
       [ 3712.632     ,  3518.592     ,   731.808     ],
       [  986.26266667,   997.93933333,   137.565     ]])

restricted lambda


array(0.40735104)

route_improvement P


array([[ 5312.        , 14992.        ,     0.        ],
       [  737.771261  ,   713.5483871 ,     0.        ],
       [  243.7       ,  1377.4       ,     0.        ],
       [ 3712.632     ,  3518.592     ,   731.808     ],
       [  986.26266667,   997.93933333,   137.565     ]])

route_improvement lambda


array(0.40735104)